<a href="https://colab.research.google.com/github/tqd3pz/ds2002-fa26/blob/main/notebooks/03-pandas-cleaning/2026-09-23%20%E2%80%94%20Cleaning%20Clinic%20%E2%80%94%20Studio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [43]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [44]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [45]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)

print("\nNull values:")
print(df.isnull().sum())

print("\nExact duplicate rows:")
print(df.duplicated().sum())

Shape: (8, 6)

Data types:
order_id      int64
item         object
category     object
qty         float64
price        object
ts           object
dtype: object

Null values:
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

Exact duplicate rows:
1


**What is wrong with this data?** List at least five specific problems:

1. there is one exact duplicate row for order 1
2. the item names are inconsistent, such as "Cheeseburger" and "cheese burger"
3. the category names are inconsistent, such as "Food" vs "food" and "RainGear" vs "rain-gear"
4. there are missing values, including a missing quantity, item name, and timestamp.
5. the price column has inconsistent formatting because some prices contain $ signs while others do not

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [46]:
removed = df.duplicated().sum()   # TODO: how many duplicates were there?
clean = df.drop_duplicates().copy() # TODO: df with duplicates dropped, copied

log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [47]:
clean['price'] = clean['price'].str.replace('$','',regex=False).str.strip().astype(float)

assert clean['price'].dtype == float
log('price','removed dollar signs/whitespace and converted price from text to float', len(clean))

[price] removed dollar signs/whitespace and converted price from text to float (7 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [48]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

missing = clean['qty'].isna().sum()    # TODO: count of NaN quantities
negative = (clean['qty'] < 0).sum()   # TODO: count of negative quantities
clean = clean[clean['qty'].notna()].copy()

# TODO: apply your decision, then log both separately
log('quantity missing', 'dropped rows with missing quantity', missing)
log('quantity negative', 'kept negative quantities as refunds', negative)

[quantity missing] dropped rows with missing quantity (1 row(s))
[quantity negative] kept negative quantities as refunds (1 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [49]:
before_count = clean['category'].nunique()
print('before:', sorted(clean['category'].unique()))

# TODO: lowercase, strip, remove punctuation
clean['category'] = clean['category'].str.lower().str.strip()
clean['category'] = clean['category'].str.replace('-', '', regex=False)

# TODO: CATEGORY_MAP = {...} for the judgment calls
CATEGORY_MAP = {
    'raingear': 'rain gear'
}

clean['category'] = clean['category'].replace(CATEGORY_MAP)

# print('after: ', sorted(clean['category'].unique()))
print('after:', sorted(clean['category'].unique()))

after_count = clean['category'].nunique()

log('categories',
    f'normalized categories from {before_count} distinct values to {after_count}',
    len(clean))

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after: ['apparel', 'food', 'merch', 'rain gear']
[categories] normalized categories from 6 distinct values to 4 (6 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [50]:
print('before:', sorted(clean['item'].dropna().unique()))

clean['item'] = clean['item'].str.lower().str.strip()

ITEM_MAP = {
    'cheese burger': 'cheeseburger'
}

clean['item'] = clean['item'].replace(ITEM_MAP)

missing_item = clean['item'].isna().sum()
clean = clean[clean['item'].notna()].copy()

print('after:', sorted(clean['item'].unique()))

log('item names', 'standardized item names and combined spelling variants', len(clean))
log('item missing', 'dropped rows with missing item name', missing_item)

before: ['Cheeseburger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'rain poncho']
after: ['cheeseburger', 'rain poncho', 'uva t-shirt']
[item names] standardized item names and combined spelling variants (5 row(s))
[item missing] dropped rows with missing item name (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [51]:
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce', format='mixed')

failed = clean['ts'].isna().sum()

clean['hour'] = clean['ts'].dt.hour

log('timestamps', 'parsed timestamps to datetime; failures converted to NaT', failed)

[timestamps] parsed timestamps to datetime; failures converted to NaT (1 row(s))


### Step 7 — prove it

**TODO:** write at least five assertions that would catch a regression in this pipeline. Then compute `revenue` and print the totals.

In [52]:
assert clean.duplicated().sum() == 0
assert clean['qty'].isna().sum() == 0
assert clean['item'].isna().sum() == 0
assert pd.api.types.is_float_dtype(clean['price'])
assert pd.api.types.is_datetime64_any_dtype(clean['ts'])
assert set(clean['category'].unique()) == {'food', 'apparel', 'rain gear'}

clean['revenue'] = clean['qty'] * clean['price']

print('rows:', len(clean))
print('units:', clean['qty'].sum())
print('revenue:', clean['revenue'].sum())
print('distinct categories:', clean['category'].nunique())

rows: 5
units: 6.0
revenue: 76.5
distinct categories: 3


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [53]:
show_log()

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,removed dollar signs/whitespace and converted ...,7
2,quantity missing,dropped rows with missing quantity,1
3,quantity negative,kept negative quantities as refunds,1
4,categories,normalized categories from 6 distinct values to 4,6
5,item names,standardized item names and combined spelling ...,5
6,item missing,dropped rows with missing item name,1
7,timestamps,parsed timestamps to datetime; failures conver...,1


**The decision that mattered most: keeping the negative quantity as a refund

**Revenue with it:** $76.50  
**Revenue without it:** $94.50

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [54]:
# Checkpoint
rows_after = len(clean)          # TODO
revenue_after = clean['revenue'].sum()       # TODO
biggest_decision = 'kept negative quantity as a refund'    # TODO: which choice moved the number most
revenue_other_way = clean.loc[clean['qty'] >= 0, 'revenue'].sum()     # TODO: the total if you had chosen differently

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)

rows after cleaning: 5
revenue: 76.5
decision that mattered: kept negative quantity as a refund
revenue the other way: 94.5
